In [64]:
import pymupdf
import re
# from PIL import Image
import numpy as np
import unicodedata
from typing import List, Dict
import json

# Extract PT

In [72]:
path = "/home/inacio/clima-amazonia/lib/boletim/6/7/v6_n7.pdf"

In [94]:
def slugify(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"[^\w\s-]", "", text).strip().lower()
    text = re.sub(r"[-\s]+", "-", text)
    return text

def extract_fields(text: str) -> Dict[str, str]:

    climatologia = re.search(r"entre\s+(\d+\s+e\s+\d+\s+mm)", text).group(1)
    min = climatologia.split()[0]
    max = climatologia.split()[2]
    observados = re.search(r"foram\s+observados\s+(\d+\s+mm)", text)
    anomalia = re.search(r"valor\s+de\s+([-\d\.]+)", text)
    classification = re.search(r"classifica\s+a\s+bacia\s+em\s+condição\s+de\s+([^\.]+)", text)
    prognostico = re.search(r"sugere\s+um\s+comportamento\s+([^\.]+)", text)
    trend = text.split("comportamento climático indica ")
    trend = trend[1].split(" ",1)[0]


    return {
        # "climatologia": climatologia.group(1) if climatologia else None,
        "min": min,
        "max": max,
        "observados": observados.group(1) if observados else None,
        "anomalia": anomalia.group(1) if anomalia else None,
        "classification": classification.group(1).strip() if classification else None,
        "prognostico": prognostico.group(1).strip() if prognostico else None,
        "trend": trend
    }

In [ ]:
def get_meta(doc, bulletin_dict):
    # Page 1
    page = doc.load_page(0)
    text = page.get_text()
    text_lines = text.splitlines()
    doi = text_lines[0].split(":")[1]
    issn = text_lines[2].split(": ")[1]
    title = " ".join(text_lines[3:-1]).replace("  ", " ")
    volume = text_lines[-1].split()[1].removesuffix(",")
    number = text_lines[-1].split()[3]
    date = " ".join(text_lines[-1].split()[5:])
    d_page = {
        'doi': doi,
        'issn': issn,
        'volume': volume,
        'number': number,
        'date': date,
        'title': title
    }
    bulletin_dict.update(d_page)
    return bulletin_dict

def get_current_conditions(doc, bulletin_dict):
    output_path = "/home/inacio/clima-amazonia/lib/test/current_conditions"
     # Page 4
    page = doc.load_page(3)
    text = page.get_text()
    inicio = r"Mapas das condições observadas de precipitação"
    fim = r"Negro na primeira semana\."
    padrao = rf"({inicio}.*?{fim})"
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    match = re.search(padrao, text, re.DOTALL)
    match.group(1).strip()
    current_conditions = {
        "text": match.group(1).strip(),
        "map_current_conditions": "map_current_conditions.png",
        "table_current_conditions": "table_current_conditions.png"
    }
    bulletin_dict["current_conditions"] = current_conditions
    # Images
    # mapa
    x0, y0, x1, y1 = 100, 400, 483, 670  
    rect = pymupdf.Rect(x0, y0, x1, y1)
    zoom = 3 
    mat = pymupdf.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{output_path}/map_current_conditions.png')
    print('Mapa de condições atuais salvo com sucesso.')
    # table
    x0, y0, x1, y1 = 75, 670, 520, 765   
    rect = pymupdf.Rect(x0, y0, x1, y1)
    mat = pymupdf.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{output_path}/table_current_conditions.png')
    print('Tabela de condições atuais salva com sucesso.')
    return bulletin_dict



def get_multimodel(doc, bulletin_dict):
    page = doc.load_page(15)
    text = page.get_text()
    text = re.sub(r"\s+", " ", text).strip()
    padrao = re.compile(
        r"(Previsão multimodelo subsazonal CPTEC/INPE-FUNCEME\s+produzida\s+em\s+"
        r"\d{2}/\d{2}/\d{4}\s+para\s+os\s+próximos\s+7\s+e\s+14\s+dias\.)",
        flags=re.IGNORECASE
    )
    match = padrao.search(text)
    title = match.group(1)
    padrao = re.compile(
        r"(A previsão multimodelo subsazonal calibrada CPTEC/INPE-FUNCEME.*?"
        r"bacias de interesse\.)",
        flags=re.IGNORECASE
    )
    match = padrao.search(text)
    texto = match.group(1)
    padrao = re.compile(
        r"(A Figura acima, apresenta o prognóstico para o intervalo.*?"
        r"bacias monitoradas\.)",
        flags=re.IGNORECASE
    )
    match = padrao.search(text)
    seven_days =match.group(1)
    page = doc.load_page(16)
    text = page.get_text()
    text = re.sub(r"\s+", " ", text).strip()
    fourteen_days = text.replace("Bacia Amazônica – CODAM Página 14 ", "").split(" 1 Abacaxis ")[0]
    multimodel = {
        "title": title,
        "text": texto,
        "seven_days": seven_days,
        "img_seven_days": "seven_days.png",
        "fourteen_days": fourteen_days,
        "img_fourteen_days": "fourteen_days.png"
    },
    bulletin_dict['multimodel'] = multimodel
    # Images
    output_path = "/home/inacio/clima-amazonia/lib/test/multimodel"
    # Images Multimodel
    page = doc.load_page(15)
    x0, y0, x1, y1 = 70, 200, 515, 620   
    rect = pymupdf.Rect(x0, y0, x1, y1)
    zoom = 3 
    mat = pymupdf.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{output_path}/seven_days.png')
    page = doc.load_page(16)
    x0, y0, x1, y1 = 70, 70, 515, 500   
    rect = pymupdf.Rect(x0, y0, x1, y1)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{output_path}/fourteen_days.png')
    return bulletin_dict

def get_anomaly(doc, bulletin_dict):
    page = doc.load_page(17)
    text = page.get_text()
    text = re.sub(r"\s+", " ", text).strip()
    padrao = re.compile(
        r"(A Tabela 1, mostra os valores de precipitação média acumulada.*?"
        r"condições em cada bacia monitorada\.)",
        flags=re.IGNORECASE
    )
    match = padrao.search(text)
    texto = match.group(1)
    padrao = re.compile(
        r"(Tabela 1. Quantis de precipitação acumulada.*?"
        r"dados MERGE/GPM – INPE/CPTEC\.)",
        flags=re.IGNORECASE
    )
    match = padrao.search(text)
    legend_table = match.group(1)
    reference = {
        "text": texto,
        "legend_table": legend_table,
        "img_reference": "reference.png"
    }
    bulletin_dict['reference'] = reference
    # Table
    zoom = 3 
    mat = pymupdf.Matrix(zoom, zoom)
    output_path = "/home/inacio/clima-amazonia/lib/test/anomaly"
    x0, y0, x1, y1 = 90, 230, 550, 690   
    rect = pymupdf.Rect(x0, y0, x1, y1)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{output_path}/reference.png')
    x0, y0, x1, y1 = 80, 415, 530, 740    
    rect = pymupdf.Rect(x0, y0, x1, y1)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{output_path}/anomaly_table.png')
    # Behavoir
    c = 1
    logo = (175.0500030517578, 776.2003173828125, 457.0, 832.2003173828125)
    output_path = "/home/inacio/clima-amazonia/lib/test/anomaly"
    for i in range(19, 23):
            page = doc.load_page(i)
            page_dict = page.get_text("dict") 
            blocks = page_dict.get("blocks", [])
            for b in blocks:
                btype = b.get("type", None)  
                bbox = b.get("bbox", None)
                if btype == 1 and bbox != logo:
                    rect = pymupdf.Rect(bbox)
                    pix = page.get_pixmap(clip=rect, dpi=200, alpha=False)
                    pix.save(f'{output_path}/chart_{c}.png')
                    c += 1
    return bulletin_dict

In [96]:
def get_analysis(doc, bulletin_dict):
    bacias = ['Bacia do Rio Branco', 'Bacia do Rio Negro','Bacia do Rio Marañon', 'Bacia do Rio Ucayali', 'Bacia do Rio Napo', 
'Curso principal do Rio Amazonas (Peru)','Bacia do Rio Javari','Bacia do Rio Içá (Putumayo)','Bacia do Rio Jutaí','Bacia do Rio Juruá',
'Bacia do Rio Japurá (Caquetá)','Bacia do Rio Tefé','Bacia do Rio Coari','Bacia do Rio Purus','Curso principal do Rio Solimões',
'Bacia dos rios Beni e Madre de Dios','Bacia do Rio Mamoré','Bacia do Rio Guaporé (Iténez)','Bacia do Rio Ji-Paraná','Bacia do Rio Aripuanã',
'Bacia do Rio Madeira','Bacias da margem esquerda do Rio Amazonas (Amazonas)','Bacia do Rio Abacaxis','Bacia do Rio Juruena','Bacia do Rio Teles Pires',
'Bacia do Rio Tapajós','Bacias da margem esquerda do Rio Amazonas (noroeste do Pará)','Bacia do Rio Curuá Una','Bacias da margem esquerda do Rio Amazonas (nordeste do PA)',
'Bacia do Rio Iriri','Bacia do Rio Xingu','Curso principal do Rio Amazonas (Brasil)']
    analysis = []
    texts = [doc.load_page(pg).get_text() for pg in range(4,15)]
    text = " ".join(texts)
    text = re.sub(r"\s*\n\s*", " ", text)
    text = re.sub(r"\s{2,}", " ", text).strip()
    for i, nome in enumerate(bacias):
        nome_regex = re.escape(nome)
        start_match = re.search(nome_regex, text)
        start = start_match.end()
        if i < len(bacias) - 1:
            next_nome = re.escape(bacias[i + 1])
            next_match = re.search(next_nome, text)
            end = next_match.start() if next_match else len(text)
        else:
            end = len(text)
        basin_text = text[start:end].strip()
        fields = extract_fields(basin_text)
        slug_name = slugify(nome)
        bacia = {
                "id": slug_name,
                "name": nome,
                # "text": basin_text,
                "mim": fields["min"],
                "max": fields["max"],
                "observados": fields["observados"],
                "anomalia": fields["anomalia"],
                "classification": fields["classification"],
                "prognostico": fields["prognostico"],
                "trend": fields['trend']
            #      "charts": {
            #         "acc": f"{slug_name}-acc.png",
            #         "ano": f"{slug_name}-ano.png"
            # }
            }
        analysis.append(bacia)
    bulletin_dict['analysis'] = analysis
    # Images
    # images = []
    # for i in analysis:
    #     id = i['id']
    #     images.append(f'{id}-acc.png')
    #     images.append(f'{id}-ano.png')
    # c = 0
    # logo = (175.0500030517578, 776.2003173828125, 457.0, 832.2003173828125)
    # output_path = '/home/inacio/clima-amazonia/lib/test/analysis/'
    # for i in range(4, 15):
    #     page = doc.load_page(i)
    #     page_dict = page.get_text("dict") 
    #     blocks = page_dict.get("blocks", [])
    #     for b in blocks:
    #         btype = b.get("type", None)  
    #         bbox = b.get("bbox", None)
    #         if btype == 1 and bbox != logo:
    #             rect = pymupdf.Rect(bbox)
    #             pix = page.get_pixmap(clip=rect, dpi=200, alpha=False)
    #             img = images[c]
    #             pix.save(f'{output_path}/{img}')
    #             c += 1
    
    return bulletin_dict

In [97]:
def get_text(path):
    bulletin_dict = {}
    doc = pymupdf.open(path)
    # bulletin_dict = get_meta(doc, bulletin_dict)
    # bulletin_dict = get_current_conditions(doc, bulletin_dict)
    bulletin_dict = get_analysis(doc, bulletin_dict)
    # bulletin_dict = get_multimodel(doc, bulletin_dict)
    # bulletin_dict = get_anomaly(doc, bulletin_dict)
    return bulletin_dict
    

bulletin_dict = get_text(path)
bulletin_dict

{'analysis': [{'id': 'bacia-do-rio-branco',
   'name': 'Bacia do Rio Branco',
   'mim': '36',
   'max': '48',
   'observados': '30 mm',
   'anomalia': '-0.9',
   'classification': 'tendência a seco',
   'prognostico': 'próximo da normalidade ou tendência a seco',
   'trend': 'elevação'},
  {'id': 'bacia-do-rio-negro',
   'name': 'Bacia do Rio Negro',
   'mim': '171',
   'max': '198',
   'observados': '186 mm',
   'anomalia': '0.0',
   'classification': 'normalidade',
   'prognostico': 'próximo da normalidade ou tendência a seco',
   'trend': 'manutenção'},
  {'id': 'bacia-do-rio-maranon',
   'name': 'Bacia do Rio Marañon',
   'mim': '158',
   'max': '180',
   'observados': '257 mm',
   'anomalia': '1.9',
   'classification': 'tendência a muito chuvoso',
   'prognostico': 'muito chuvoso ou tendência a muito chuvoso',
   'trend': 'elevação'},
  {'id': 'bacia-do-rio-ucayali',
   'name': 'Bacia do Rio Ucayali',
   'mim': '181',
   'max': '203',
   'observados': '271 mm',
   'anomalia': '1.

In [40]:
with open('/home/inacio/clima-amazonia/lib/test/pt.json', 'w') as f:
    json.dump(bulletin_dict, f, ensure_ascii=False, indent=3)

In [74]:
t = '36 e 48 mm'
t.split()

['36', 'e', '48', 'mm']

# Extract EN

In [2]:
path = "/home/inacio/clima-amazonia/lib/boletim/6/7/BHA_EN_20260218.pdf"

In [3]:
def get_meta(doc, bulletin_dict):
    # Page 1
    page = doc.load_page(0)
    text = page.get_text()
    volume = text.split("Volume ")[1].split(", ",1)[0]
    number = text.split('Number ')[1].rsplit()[0]
    date = text.split('Manaus, ')[1].removesuffix('\n')
    d_page = {
        'volume': volume,
        'number': number,
        'date': date,
    }
    bulletin_dict.update(d_page)
    return bulletin_dict

In [4]:
def get_current_conditions(doc, bulletin_dict):
    output_path = "/home/inacio/clima-amazonia/lib/test/current_conditions"
     # Page 4
    page = doc.load_page(3)
    text = page.get_text()
    inicio = r"Maps of observed precipitation conditions"
    fim = r"Japurá and Negro river basins in the first week\."
    padrao = rf"({inicio}.*?{fim})"
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    match = re.search(padrao, text, re.DOTALL)
    match.group(1).strip()
    current_conditions = {
        "text": match.group(1).strip(),
        "map_current_conditions": "map_current_conditions.png",
        "table_current_conditions": "table_current_conditions.png"
    }
    bulletin_dict["current_conditions"] = current_conditions
    # Images
    # mapa
    x0, y0, x1, y1 = 100, 400, 483, 670  
    rect = pymupdf.Rect(x0, y0, x1, y1)
    zoom = 3 
    mat = pymupdf.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{output_path}/map_current_conditions.png')
    print('Mapa de condições atuais salvo com sucesso.')
    # table
    x0, y0, x1, y1 = 75, 670, 520, 765   
    rect = pymupdf.Rect(x0, y0, x1, y1)
    mat = pymupdf.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{output_path}/table_current_conditions.png')
    print('Tabela de condições atuais salva com sucesso.')
    return bulletin_dict

In [ ]:
def extract_fields(text: str) -> Dict[str, str]:

    pattern = r'(\d+(?:\.\d+)?)\s*(?:and|e)\s*(\d+(?:\.\d+)?)\s*mm'
    match = re.search(pattern, text, re.IGNORECASE)
    climatologia = f"{match.group(1)} and {match.group(2)} mm"
    observados = re.search(r"foram\s+observados\s+(\d+\s+mm)", text)
    anomalia = re.search(r"valor\s+de\s+([-\d\.]+)", text)
    classification = re.search(r"classifica\s+a\s+bacia\s+em\s+condição\s+de\s+([^\.]+)", text)
    prognostico = re.search(r"sugere\s+um\s+comportamento\s+([^\.]+)", text)

    return {
        "climatologia": climatologia,
        "observados": observados.group(1) if observados else None,
        "anomalia": anomalia.group(1) if anomalia else None,
        "classification": classification.group(1).strip() if classification else None,
        "prognostico": prognostico.group(1).strip() if prognostico else None,
    }

In [ ]:

def get_analysis(doc, bulletin_dict):
    bacias = ['Branco River Basin', 
    'Negro River Basin',
    'Marañon River Basin',
    'Ucayali River Basin',
    'Napo River Basin',
    'The main course of the Amazon River (Peru)',
    'Javari River Basin',
    'Putumayo and Içá Rivers Basins',
    'Jutaí River Basin',
    'Juruá River Basin',
    'Caquetá and Japurá Rivers Basins',
    'Tefé River Basin',
    'Coari River Basin',
    'Purus River Basin',
    'The main course of the Solimões River',
    'Madre de Dios and Beni Rivers Basins',
    'Mamoré River Basin',
    'Guaporé and Intenéz Rivers Basins',
    'Ji-Paraná River Basin',
    'Aripuanã River Basin',
    'Madeira River Basin',
    'Basins on the left bank of the Amazon River (Amazonas State)',
    'Abacaxis River Basin',
    'Juruena River Basin',
    'Teles Pires River Basin',
    'Tapajós River Basin',
    'Basins on the left bank of the Amazon River (northwest of the Pará State)',
    'Curuá Una River Basin',
    'Basins on the left bank of the Amazon River (north-estern of the Pará State)',
    'Iriri River Basin',
    'Xingu River Basin',
    'The main course of the Amazon River (Brazil)']      
    analysis = []
    texts = [doc.load_page(pg).get_text() for pg in range(4,15)]
    text = " ".join(texts)
    text = re.sub(r"\s*\n\s*", " ", text)
    text = re.sub(r"\s{2,}", " ", text).strip()
    for i, nome in enumerate(bacias):
        nome_regex = re.escape(nome)
        start_match = re.search(nome_regex, text)
        start = start_match.end()
        if i < len(bacias) - 1:
            next_nome = re.escape(bacias[i + 1])
            next_match = re.search(next_nome, text)
            end = next_match.start() if next_match else len(text)
        else:
            end = len(text)
        basin_text = text[start:end].strip()
        fields = extract_fields(basin_text)
        slug_name = slugify(nome)
        bacia = {
                "id": slug_name,
                "name": nome,
                "text": basin_text,
                "climatologia": fields["climatologia"],
                "observados": fields["observados"],
                "anomalia": fields["anomalia"],
                "classification": fields["classification"],
                "prognostico": fields["prognostico"],
                 "charts": {
                    "acc": f"{slug_name}-acc.png",
                    "ano": f"{slug_name}-ano.png"
            }
            }
        analysis.append(bacia)
    bulletin_dict['analysis'] = analysis
    # Images
    images = []
    for i in analysis:
        id = i['id']
        images.append(f'{id}-acc.png')
        images.append(f'{id}-ano.png')
    c = 0
    logo = (172.0, 776.2003173828125, 442.1990051269531, 829.7503662109375)
    output_path = '/home/inacio/clima-amazonia/lib/test/analysis/'
    for i in range(4, 15):
        page = doc.load_page(i)
        page_dict = page.get_text("dict") 
        blocks = page_dict.get("blocks", [])
        for b in blocks:
            btype = b.get("type", None)  
            bbox = b.get("bbox", None)
            if btype == 1 and bbox != logo:
                rect = pymupdf.Rect(bbox)
                pix = page.get_pixmap(clip=rect, dpi=200, alpha=False)
                img = images[c]
                pix.save(f'{output_path}/{img}')
                c += 1
    
    return bulletin_dict

In [ ]:
def get_multimodel(doc, bulletin_dict):
    page = doc.load_page(15)
    # Title
    text = page.get_text()
    text = re.sub(r"\s+", " ", text).strip()
    pattern = r"A multi-model sub-seasonal forecast CPTEC/INPE-FUNCEME produced on \d{2}/\d{2}/\d{4}\s+for the next 7 and 14 days\."
    match = re.search(pattern, text)
    title = match.group()
    
    pattern = r"(The figure above shows the forecast for the 7-day interval.*?other monitored basins\.)"
    match = re.search(pattern, text)
    seven_days = match.group()
    
    page = doc.load_page(16)
    text = page.get_text()
    clean_text = re.sub(r'\s+', ' ', text)
    pattern = r"(The figure above shows the prognosis for the 14-day interval.*?Teles Pires river basins\.)"
    match = re.search(pattern, clean_text)
    fourteen_days = match.group(1)
    
    multimodel = {
        "title": title,
        # "text": texto,
        "seven_days": seven_days,
        "img_seven_days": "seven_days.png",
        "fourteen_days": fourteen_days,
        "img_fourteen_days": "fourteen_days.png"
    },
    bulletin_dict['multimodel'] = multimodel
    # Images
    output_path = "/home/inacio/clima-amazonia/lib/test/multimodel"
    # Images Multimodel
    page = doc.load_page(15)
    x0, y0, x1, y1 = 70, 200, 515, 620   
    rect = pymupdf.Rect(x0, y0, x1, y1)
    zoom = 3 
    mat = pymupdf.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{output_path}/seven_days.png')
    page = doc.load_page(16)
    x0, y0, x1, y1 = 70, 70, 515, 500   
    rect = pymupdf.Rect(x0, y0, x1, y1)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{output_path}/fourteen_days.png')
    return bulletin_dict


In [56]:
def get_anomaly(doc, bulletin_dict):
    page = doc.load_page(17)
    text = page.get_text()
    pattern = r"Table 1\. Accumulated precipitation quantiles \(mm\) in 30 days \([^)]+\)"
    match = re.search(pattern, text)
    legend = match.group()
    reference = {
        "legend_table": legend
    }
    bulletin_dict['reference'] = reference
    # Table
    zoom = 3 
    mat = pymupdf.Matrix(zoom, zoom)
    output_path = "/home/inacio/clima-amazonia/lib/test/anomaly"
    x0, y0, x1, y1 = 90, 230, 550, 690   
    rect = pymupdf.Rect(x0, y0, x1, y1)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{output_path}/reference.png')
    page = doc.load_page(18)
    x0, y0, x1, y1 = 80, 405, 540, 760    
    rect = pymupdf.Rect(x0, y0, x1, y1)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{output_path}/anomaly_table.png')
    # Behavoir
    c = 1
    logo = (175.0500030517578, 776.2003173828125, 457.0, 832.2003173828125)
    output_path = "/home/inacio/clima-amazonia/lib/test/anomaly"
    for i in range(19, 23):
            page = doc.load_page(i)
            page_dict = page.get_text("dict") 
            blocks = page_dict.get("blocks", [])
            for b in blocks:
                btype = b.get("type", None)  
                bbox = b.get("bbox", None)
                if btype == 1 and bbox[1] < 760:
                    rect = pymupdf.Rect(bbox)
                    pix = page.get_pixmap(clip=rect, dpi=200, alpha=False)
                    pix.save(f'{output_path}/chart_{c}.png')
                    c += 1
    return bulletin_dict

In [ ]:

bulletin_dict = {}
doc = pymupdf.open(path)
bulletin_dict = get_meta(doc, bulletin_dict)
bulletin_dict = get_current_conditions(doc, bulletin_dict)
bulletin_dict = get_analysis(doc, bulletin_dict)
bulletin_dict = get_multimodel(doc, bulletin_dict)
bulletin_dict = get_anomaly(doc, bulletin_dict)
bulletin_dict

In [58]:
with open('/home/inacio/clima-amazonia/lib/test/en.json', 'w') as f:
    json.dump(bulletin_dict, f, ensure_ascii=False, indent=3)

In [63]:
bulletin_dict = get_analysis(doc, bulletin_dict)
bulletin_dict

{'volume': '4',
 'number': '07',
 'date': 'February 18, 2026',
 'current_conditions': {'text': 'Maps of observed precipitation conditions, individual graphics by basin are produced from MERGE/GPM data generated by INPE/CPTEC, considering the period from de 2000 a 2025. Between January 20 to February 18, 2026, below-average rainfall caused precipitation deficits along the main course of the Amazon River in Brazilian territory, the watersheds of the Abacaxis, Branco, Coari, Curuá Una, Guaporé, Iriri, Jutaí, Mamoré, the left bank basins of the Amazon River in the northeast and northwest of the state of Pará, Purus, Tapajós, Xingu, and the main course of the Solimões River; rainfall above climatological norms was recorded on the main course of the Amazon River in Peruvian territory and in the basins of the Içá, Javari, Marañon, Napo, and Ucayali rivers; rainfall close to normal recorded over the Aripuanã, Beni, Japurá, Ji-Paraná, Juruá, Juruena, and Madeira river basins, basins on the left

In [53]:
logo[1] > 760

True

In [51]:
rect = pymupdf.Rect(172, 760, 442.1990051269531, 829.7503662109375)
pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
pix.save(f'{output_path}/test.png')

In [36]:
page = doc.load_page(18)
x0, y0, x1, y1 = 80, 405, 540, 760    
rect = pymupdf.Rect(x0, y0, x1, y1)
zoom = 3 
mat = pymupdf.Matrix(zoom, zoom)
pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
output_path = "/home/inacio/clima-amazonia/lib/test/anomaly"
pix.save(f'{output_path}/anomaly_table.png')

In [19]:
pattern = r"Table 1\. Accumulated precipitation quantiles \(mm\) in 30 days \([^)]+\)"
match = re.search(pattern, text)

In [21]:
match.group()

'Table 1. Accumulated precipitation quantiles (mm) in 30 days (20 de January a 18 de February)'